In [1]:
import os

os.environ["RAY_DEDUP_LOGS"] = "0"
os.environ["RAY_DISABLE_METRICS"] = "1"
os.environ["RAY_DISABLE_METRICS_EXPORT"] = "1"

In [2]:

import holoviews as hv
import polars as pl
from sklearn.metrics import accuracy_score

from autotsc import utils
from autotsc.models import *
from autotsc.utils import *
from autotsc.utils import load_dataset

2025-11-27 14:08:08.090883: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
X_train, y_train, X_test, y_test = load_dataset("CricketZ")
# X_train, y_train, X_test, y_test = load_dataset("MelbournePedestrian")
# X_train, y_train, X_test, y_test = load_dataset("Worms")
# X_train, y_train, X_test, y_test = load_dataset("PhalangesOutlinesCorrect")
# X_train, y_train, X_test, y_test = load_dataset("Lightning2")
# X_train, y_train, X_test, y_test = load_dataset("CricketY")
# X_train, y_train, X_test, y_test = load_dataset("OSULeaf")
# X_train, y_train, X_test, y_test = load_dataset("Wine")
# X_train, y_train, X_test, y_test = load_dataset("FiftyWords")
X_train, y_train, X_test, y_test = load_dataset("Haptics")


# X_train, y_train, X_test, y_test = load_dataset("InlineSkate")
# X_train, y_train, X_test, y_test = load_dataset("SemgHandSubjectCh2")

In [4]:
#

In [ ]:
with utils.ray_init_or_reuse(num_cpus=24, resources={"meta": 100}, ignore_reinit_error=True):
    model = AutoTSCModel(model_selection="fast", verbose=1)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print("----- CA", accuracy_score(y_test, pred))

2025-11-27 14:08:13,840	INFO worker.py:2012 -- Started a local Ray instance.
/home/gasper_p/workspace/repos/AutoTSC/.venv/lib/python3.12/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
(ray_run_fit_predict_proba_wrapper pid=3021479) 2025-11-27 14:08:18.700822: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
(ray_run_fit_predict_proba_wrapper pid=3021479) To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
(ray_run_fit_predict_proba_wrapper pid=3021484) 2025-11-27 14:08:18.700796: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow

In [ ]:
summary = model.summary()
summary

In [ ]:
summary.sort("validation_accuracy").tail(1)["model_id"].item()

In [ ]:
test_accs = []
preds = model.predict_per_model(X_test)
for m, p in preds.items():
    acc = accuracy_score(y_test, p)
    test_accs.append(
        {
            "model_id": m,
            "test_accuracy": acc,
        }
    )
test_accs = pl.DataFrame(test_accs)
summary = summary.join(test_accs, on="model_id")

In [ ]:
pl.Config.set_fmt_str_lengths(500)
for r in summary.sort("validation_accuracy").iter_rows():
    print(r)

In [ ]:
# Create the scatter plot
scatter = summary.hvplot.scatter(
    x="validation_accuracy",
    y="test_accuracy",
    c="stacking_level",
    hover_cols=["model_id", "validation_accuracy", "test_accuracy", "stacking_level"],
    cmap="viridis",
    size=80,
    alpha=0.7,
    title="Validation vs Test Accuracy per Model",
    xlabel="Validation Accuracy",
    ylabel="Test Accuracy",
    width=700,
    height=600,
    grid=True,  # Add grid here
)

# Add text labels for model_id
labels = hv.Labels(
    summary.select(["validation_accuracy", "test_accuracy", "model_id"]).to_pandas(),
    kdims=["validation_accuracy", "test_accuracy"],
    vdims=["model_id"],
).opts(
    text_font_size="8pt",
    text_align="left",
    text_baseline="bottom",
    xoffset=0.0005,  # slight offset so text doesn't overlap point
    yoffset=0.0005,
)

# Add diagonal line
min_val = min(summary["validation_accuracy"].min(), summary["test_accuracy"].min())
max_val = max(summary["validation_accuracy"].max(), summary["test_accuracy"].max())
diagonal = hv.Curve([(min_val, min_val), (max_val, max_val)]).opts(
    color="gray", line_dash="dashed", line_width=1
)

# Combine all
plot = scatter * labels * diagonal
plot